In [26]:
import pandas as pd
import numpy as np
import re
import ast
import os

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler, MinMaxScaler
from scipy.stats import zscore

In [2]:
dataset_paintings = pd.read_csv("processed_data/extended_paintings_dataset_BIRD.csv", encoding="utf-8-sig")
dataset = pd.read_csv("processed_data/pre_processed_ds.csv", encoding="utf-8-sig")

# inject paintings information into dataset
dataset_extended = dataset.merge(dataset_paintings[['paintingId', 'painting_style', 'painting_genre']], 
                                 left_on='paintingId', right_on='paintingId', how='left')

dataset_extended = dataset_extended.dropna(subset=['painting_style', 'painting_genre'])

# check the type convert to float
dataset_extended['duration_seconds'] = (dataset_extended['duration_seconds']
    .astype(str)                 
    .str.replace(',', '.')           
    .astype(float))

# remove negative samples
dataset_extended = dataset_extended[dataset_extended['duration_seconds'] >= 0]

# calcualte the target engagement_level
dataset_extended["total_view_time"] = (dataset_extended
    .groupby(["user_id", "paintingId"])["duration_seconds"]
    .transform("sum"))

dataset_extended["num_visits"] = (dataset_extended
    .groupby(["user_id", "paintingId"])["duration_seconds"]
    .transform("count"))

dataset_extended["num_revisits"] = (dataset_extended["num_visits"] - 1).clip(lower=0)

dataset_extended["max_single_view"] = (dataset_extended
    .groupby(["user_id", "paintingId"])["duration_seconds"]
    .transform("max"))

# remove repeating rows
dataset_extended = (dataset_extended
    .drop_duplicates(subset=["user_id", "paintingId"])
    .reset_index(drop=True))

# delete auxuliary column
dataset_extended = dataset_extended.drop(columns=["num_visits"])

# log transform
cols = ['total_view_time', 'num_revisits', 'max_single_view']

for c in cols:
    dataset_extended[f'log_{c}'] = np.log1p(dataset_extended[c])

# prepare the engagement level feature
dataset_extended["pct_rank"] = (dataset_extended.groupby("user_id")["duration_seconds"].rank(pct=True))


# category boundaries
conditions = [
    dataset_extended["pct_rank"] <= 0.25,

    (dataset_extended["pct_rank"] > 0.25) &
    (dataset_extended["pct_rank"] <= 0.50),

    (dataset_extended["pct_rank"] > 0.50) &
    (dataset_extended["pct_rank"] <= 0.75),

    dataset_extended["pct_rank"] > 0.75
]

# encoded engagement levels
labels = [0, 1, 2, 3]

dataset_extended["engagement_level"] = np.select(conditions, labels, default=-1).astype(int)
dataset_extended["interest_binary"] = (dataset_extended["engagement_level"] >= 2).astype(int)
dataset_extended = dataset_extended.drop("pct_rank", axis=1)

# delete or rows with NaN values
dataset_extended = dataset_extended.dropna()

In [3]:
dataset_extended

,floorNumber_start,paintingId,user_id,duration_seconds,painting_name,artist,date,art_knowledge,art_involvement,visit_rate,painting_style,painting_genre,total_view_time,num_revisits,max_single_view,log_total_view_time,log_num_revisits,log_max_single_view,engagement_level,interest_binary
0,0,0_0575,User_1,2.28,Vues d'architectures 5,Lucien Hervé,Undated,Low level,No involvement,rarely or never,Pop Art,"['Landscape', 'Cityscape']",2.28,0,2.28,1.187843,0.000000,1.187843,0,0
1,0,0_0577,User_1,12.23,"Assemblage Konshie II, polish village",Franck Stella,1971,Low level,No involvement,rarely or never,Romanticism,"['History painting', 'Mythological Painting']",12.23,0,12.23,2.582487,0.000000,2.582487,2,1
2,0,0_5832,User_1,6.34,L'enfance de Jupiter,Ker-Xavier Roussel,1940,Low level,No involvement,rarely or never,Modern Art,"['Landscape', 'Cityscape']",6.34,0,6.34,1.993339,0.000000,1.993339,1,0
3,0,0_0668,User_1,14.21,Le Grand Char,Frédéric Coché,2010,Low level,No involvement,rarely or never,Rococo,"['Landscape', 'Nature morte']",17.98,1,14.21,2.943386,0.693147,2.721953,2,1
4,0,0_0670,User_1,6.27,Anamorphose,Frédéric Coché,2010,Low level,No involvement,rarely or never,Rococo,"['Portrait', 'Nude painting']",6.27,0,6.27,1.983756,0.000000,1.983756,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5094,1,1_0356,User_51,6.00,Les patineurs,Jan Josephsz van Goyen,1649,General level,Art student,A few times a year,Baroque,"['Landscape', 'Nature morte']",6.00,0,6.00,1.945910,0.000000,1.945910,2,1
5095,1,1_0341,User_51,31.57,Marine,Cavalier Tempesta/Pieter Mulier,After 1660,General level,Art student,A few times a year,Baroque,"['Landscape', 'Nature morte']",31.57,0,31.57,3.483392,0.000000,3.483392,3,1
5096,1,1_0379,User_51,8.44,La Transfiguration,Pierre-Paul Rubens,1604-1605,General level,Art student,A few times a year,Baroque,"['History painting', 'Religious painting']",8.44,0,8.44,2.244956,0.000000,2.244956,2,1
5097,1,1_0365,User_51,13.10,Archange,Il Volterrano/ Franceschini Baldassare,After 1650,General level,Art student,A few times a year,Baroque,"['History painting', 'Religious painting']",13.10,0,13.10,2.646175,0.000000,2.646175,3,1


In [4]:
# prepreocess the date feature
def parse_artwork_date(date_str):
    """
    Converts heterogeneous artwork dates into standardized representation:
    
    - rep_year       : representative year, 4 digit number
    - range_width    : uncertainty/range width, if no single year is known
    - is_uncertain   : binary flag, if exact year known then 0 otherwise 1
    
    """

    if pd.isna(date_str):
        return pd.Series([np.nan, np.nan, 1])

    text = str(date_str).strip().lower()

    
    # undated
    if "undated" in text:
        return pd.Series([np.nan, np.nan, 1])

    
    # features containing century 
    century_match = re.search(r'(\d+)(st|nd|rd|th)\s+century', text)

    if century_match:

        century = int(century_match.group(1))

        century_start = (century - 1) * 100
        century_end = century_start + 99

        # Default full century
        start = century_start
        end = century_end

        # First quarter
        if "first quarter" in text or "1st quarter" in text:
            end = century_start + 24

        # Last quarter
        elif "last quarter" in text:
            start = century_start + 75

        # First half
        elif "first half" in text or "1st half" in text:
            end = century_start + 49

        # Second half
        elif "second half" in text:
            start = century_start + 50

        # Early
        elif "early" in text:
            end = century_start + 33

        # Mid
        elif "mid" in text or "middle" in text:
            start = century_start + 33
            end = century_start + 66

        # Late
        elif "late" in text:
            start = century_start + 66

        rep_year = (start + end) / 2
        range_width = end - start

        return pd.Series([rep_year, range_width, 1])

    
    # extract years
    years = re.findall(r'\d{4}', text)

    # Fix malformed dates like 1640-4650
    years = [int(y) for y in years if 1000 <= int(y) <= 2100]

    if len(years) == 0:
        return pd.Series([np.nan, np.nan, 1])


    # SINGLE YEAR
    if len(years) == 1:

        year = years[0]

        # before
        if "before" in text:
            return pd.Series([year - 10, 20, 1])

        # after
        if "after" in text:
            return pd.Series([year + 10, 20, 1])

        # circa
        if "circa" in text:
            return pd.Series([year, 10, 1])

        # exact year
        return pd.Series([year, 0, 0])


    # multiple years/ranges
    start = min(years)
    end = max(years)

    rep_year = (start + end) / 2
    range_width = end - start

    # Circa ranges are uncertain
    if "circa" in text:
        return pd.Series([rep_year, range_width, 1])

    # Between
    if "between" in text:
        return pd.Series([rep_year, range_width, 1])

    # Standard ranges
    return pd.Series([rep_year, range_width, 0])


# insert into the dataframe
dataset_extended[["rep_year", "range_width", "is_uncertain"]] = dataset_extended["date"].apply(parse_artwork_date)


# process missing values
# Replace NaN years with median
dataset_extended["rep_year"] = dataset_extended["rep_year"].fillna(dataset_extended["rep_year"].median())

# Replace NaN ranges with large uncertainty
dataset_extended["range_width"] = dataset_extended["range_width"].fillna(dataset_extended["range_width"].median())

In [5]:
# unify representation of all the genres
def clean_genres(genres):

    cleaned = []

    for g in genres:

        g = str(g)

        # remove extra quotes
        g = g.replace("'", "")
        g = g.replace('"', "")

        # strip whitespace
        g = g.strip()

        # normalize capitalization
        g = g.title()

        cleaned.append(g)

    return cleaned

In [6]:
# preparing the data for training
# Integer Encoding
user_encoder = LabelEncoder()
dataset_extended["user_id_encoded"] = user_encoder.fit_transform(dataset_extended["user_id"])

painting_encoder = LabelEncoder()
dataset_extended["paintingId_encoded"] = painting_encoder.fit_transform(dataset_extended["paintingId"])

artist_encoder = LabelEncoder()
dataset_extended["artist_encoded"] = artist_encoder.fit_transform(dataset_extended["artist"])


def one_hot_encode_column(df, column, prefix=None):
        
    if prefix is None:
        prefix = column

    onehot = pd.get_dummies(df[column], prefix=prefix)
    return onehot.astype(int)

style_onehot = one_hot_encode_column(dataset_extended, column="painting_style", prefix="style")
involvement_onehot = one_hot_encode_column(dataset_extended, column="art_involvement", prefix="inv")


# Ordinal_mappings
art_know_scaler = MinMaxScaler()
visit_scaler = MinMaxScaler()
knowledge_map = {"No knowledge": 0, "Low level": 1, "General level": 2, "Specialist/expert": 3, "Art lover": 4}
visit_map = {"rarely or never": 0, "A few times a year": 1, "One or more times a month": 2}

dataset_extended["art_knowledge_enc"] = dataset_extended["art_knowledge"].map(knowledge_map)
dataset_extended["visit_rate_enc"] = dataset_extended["visit_rate"].map(visit_map)

dataset_extended["art_knowledge_enc"] = art_know_scaler.fit_transform(dataset_extended[["art_knowledge_enc"]])
dataset_extended["visit_rate_enc"] = visit_scaler.fit_transform(dataset_extended[["visit_rate_enc"]])


def encode_painting_genres(df, column="painting_genre"):

    genres = df[column].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    genres = genres.apply(clean_genres)

    mlb = MultiLabelBinarizer()

    genre_multihot = pd.DataFrame(
        mlb.fit_transform(genres),
        columns=[f"genre_{c}" for c in mlb.classes_],
        index=df.index
    )

    return genre_multihot, mlb

genre_multihot, mlb = encode_painting_genres(dataset_extended)
dataset_extended = pd.concat([dataset_extended, genre_multihot], axis=1)


# Normalization
scaler = StandardScaler()
numeric_features = ["duration_seconds", "rep_year", "range_width"]
dataset_extended[numeric_features] = scaler.fit_transform(dataset_extended[numeric_features])


# Normalization target
log_cols = ['log_total_view_time', 'log_num_revisits', 'log_max_single_view']
for col in log_cols:
    dataset_extended[f'z_{col}'] = zscore(dataset_extended[col], nan_policy='omit')

dataset_extended['engagement_score'] = (0.7 * dataset_extended['z_log_total_view_time'] +
    0.15 * dataset_extended['z_log_num_revisits'] + 0.15 * dataset_extended['z_log_max_single_view'])


# Combine all processed features
final_df = pd.concat(
    [
        dataset_extended[
            [
                "user_id_encoded",
                "paintingId_encoded",
                "art_knowledge_enc",
                "visit_rate_enc",
                "artist_encoded",
                "duration_seconds",
                "rep_year",
                "range_width",
                "engagement_level",
                "engagement_score",
                "interest_binary",
                "painting_genre"
            ]
        ],
        involvement_onehot,
        style_onehot,
        genre_multihot
    ],
    axis=1
)

final_df.to_csv("processed_data/ml_ready_dataset.csv", index=False, encoding="utf-8-sig")

   user_id_encoded  paintingId_encoded  art_knowledge_enc  visit_rate_enc  \
0                0                  23               0.25             0.0   
1                0                  24               0.25             0.0   
2                0                 127               0.25             0.0   
3                0                  25               0.25             0.0   
4                0                  26               0.25             0.0   

   artist_encoded  duration_seconds  rep_year  range_width  engagement_level  \
0             182         -0.387664 -0.063548    -0.458897                 0   
1              80         -0.110161  1.456228    -0.501814                 2   
2             168         -0.274431  1.243528    -0.501814                 1   
3              90         -0.054939  1.723819    -0.501814                 2   
4              90         -0.276384  1.723819    -0.501814                 1   

   engagement_score  ...  genre_History Painting genre_I